# 🐱 vs 🐶 Cat vs Dog Classifier in PyTorch

This Google Colab notebook provides a **complete, production-grade PyTorch implementation** of the Cat vs Dog classification project (converted 1:1 from TensorFlow/Keras).

### 📌 What This Notebook Covers:
1. **Automatic Dataset Extraction**: Safe extraction of `Project CS (2).zip` or local directories.
2. **Data Pipeline**: PyTorch `torchvision.transforms`, `ImageFolder`, and GPU-accelerated `DataLoader`.
3. **Exact Neural Network Architecture**: Faithful translation of the Keras Sequential model (Flatten $\to$ Dense 228 $\to$ Dropout 0.2 $\to$ Dense 64 $\to$ Dense 1 Sigmoid).
4. **Training & Metrics Loop**: Adam optimizer, Binary Cross-Entropy loss, 40 epochs, training accuracy tracking.
5. **Visualization**: Epoch vs. Loss and Accuracy curves with `matplotlib`.
6. **Model Serialization**: Saving & loading `.pth` state dict and full model checkpoint.
7. **Test Inference**: Single/multi-image prediction with visual display and confidence score.
8. **[Bonus] CNN Architecture Comparison**: Educational upgrade showing how a Convolutional Neural Network improves generalization.

In [ ]:
# CELL 1: Environment Setup & Safe ZIP Extraction
import os
import zipfile

# Extraction destination
extract_path = '/Project_SC'

# Candidate zip file locations in Google Colab
candidate_zip_paths = [
    '/Project CS (2).zip',
    '/content/Project CS (2).zip',
    '/Project CS.zip',
    '/content/Project CS.zip',
    'Project CS (2).zip',
    'Project CS.zip'
]

zip_path = None
for p in candidate_zip_paths:
    if os.path.exists(p):
        zip_path = p
        break

# Fallback: search /content/ or / for any relevant zip file
if zip_path is None:
    for search_dir in ['/content', '/']:
        if os.path.exists(search_dir):
            for f in os.listdir(search_dir):
                if f.endswith('.zip') and ('Project' in f or 'CS' in f or 'cat' in f.lower()):
                    zip_path = os.path.join(search_dir, f)
                    break
            if zip_path:
                break

# Unzip safely if zip exists
if zip_path and os.path.exists(zip_path):
    print(f"Found ZIP archive at: {zip_path}")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"Folder connected and successfully extracted to: {extract_path}")
elif os.path.exists(os.path.join(extract_path, 'Project CS', 'Train')):
    print(f"Dataset already extracted and verified at: {extract_path}/Project CS/Train")
elif os.path.exists('/content/Project_SC/Project CS/Train'):
    extract_path = '/content/Project_SC'
    print(f"Dataset verified at: {extract_path}/Project CS/Train")
else:
    print(f"Notice: Neither ZIP file nor extracted folder found automatically.")
    print(f"Please upload 'Project CS (2).zip' to Colab via the Files sidebar or run the cell after uploading.")


In [ ]:
# CELL 2: PyTorch Dataset, Model Architecture, Training, and Evaluation
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# 1. Configure Hardware Device (GPU / CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device} (CUDA available: {torch.cuda.is_available()})")

# 2. Locate Training Directory
train_dir = '/Project_SC/Project CS/Train'
if not os.path.exists(train_dir) and os.path.exists('/content/Project_SC/Project CS/Train'):
    train_dir = '/content/Project_SC/Project CS/Train'
elif not os.path.exists(train_dir) and os.path.exists('Project_SC/Project CS/Train'):
    train_dir = 'Project_SC/Project CS/Train'

print(f"Training directory: {train_dir}")

# 3. Data Transformations (Matches Keras image_size=(224, 224) and normalization images/255.0)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()  # Automatically converts PIL image [0, 255] to float32 [0.0, 1.0]
])

# Load training dataset via PyTorch ImageFolder
train_dataset = datasets.ImageFolder(root=train_dir, transform=transform)
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2 if os.name != 'nt' else 0,
    pin_memory=torch.cuda.is_available()
)

print(f"Found {len(train_dataset)} files belonging to {len(train_dataset.classes)} classes: {train_dataset.classes}")
print(f"Class to index mapping: {train_dataset.class_to_idx}")

# 4. Build Exact PyTorch Sequential Model Architecture
# Keras counterpart:
#   Input(shape=(224, 224, 3))
#   Flatten() -> 3 * 224 * 224 = 150,528
#   Dense(228, activation='relu')
#   Dropout(0.2)
#   Dense(64, activation='relu')
#   Dense(1, activation='sigmoid')
class CatDogClassifier(nn.Module):
    def __init__(self):
        super(CatDogClassifier, self).__init__()
        self.flatten = nn.Flatten()
        self.network = nn.Sequential(
            nn.Linear(3 * 224 * 224, 228),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(228, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)

model = CatDogClassifier().to(device)
print("\n--- PyTorch Model Architecture ---")
print(model)

# 5. Define Loss Function and Optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 6. Train the Model for 40 Epochs
epochs = 40
history = {'loss': [], 'accuracy': []}

print(f"\n--- Training for {epochs} Epochs ---")
for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)  # Reshape to (batch_size, 1)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        predictions = (outputs >= 0.5).float()
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
        
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    history['loss'].append(epoch_loss)
    history['accuracy'].append(epoch_acc)
    
    print(f"Epoch {epoch:02d}/{epochs:02d} - loss: {epoch_loss:.4f} - accuracy: {epoch_acc:.4f}")

# 7. Evaluate the Model on the Training Dataset
model.eval()
eval_loss = 0.0
eval_correct = 0
eval_total = 0

with torch.no_grad():
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        eval_loss += loss.item() * images.size(0)
        predictions = (outputs >= 0.5).float()
        eval_correct += (predictions == labels).sum().item()
        eval_total += labels.size(0)

final_loss = eval_loss / eval_total
final_acc = eval_correct / eval_total
print(f"\n--- Final Evaluation ---")
print(f"Loss: {final_loss:.4f}")
print(f"Accuracy: {final_acc:.4f}")

# 8. Visualize Training Curves (Loss & Accuracy)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs + 1), history['loss'], 'b-o', label='Training Loss')
plt.title('Training Loss vs. Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (Binary Cross-Entropy)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, epochs + 1), history['accuracy'], 'g-o', label='Training Accuracy')
plt.title('Training Accuracy vs. Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim(0, 1.05)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# CELL 3: Save, Reload, and Test Inference on Unseen Images
import os
import torch
from PIL import Image
import matplotlib.pyplot as plt

# 1. Save Model Checkpoint (State Dict - PyTorch Gold Standard)
weights_path = 'cat_vs_dog.pth'
torch.save(model.state_dict(), weights_path)
print(f"Model weights saved successfully to: {weights_path}")

# Optional: Save full model object as well
torch.save(model, 'cat_vs_dog_full.pth')
print(f"Full model saved to: cat_vs_dog_full.pth")

# 2. Reload Saved Model to Verify Reproducibility
loaded_model = CatDogClassifier().to(device)
loaded_model.load_state_dict(torch.load(weights_path, map_location=device))
loaded_model.eval()
print(f"Successfully loaded and verified weights from {weights_path}")

# 3. Locate Test Directory
test_dir = '/Project_SC/Project CS/Test'
if not os.path.exists(test_dir) and os.path.exists('/content/Project_SC/Project CS/Test'):
    test_dir = '/content/Project_SC/Project CS/Test'
elif not os.path.exists(test_dir) and os.path.exists('Project_SC/Project CS/Test'):
    test_dir = 'Project_SC/Project CS/Test'

# Collect all images in the Test directory
test_images = []
if os.path.exists(test_dir):
    for root, dirs, files in os.walk(test_dir):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                test_images.append(os.path.join(root, file))

if not test_images:
    print(f"No test images found in {test_dir}. Please verify test folder location.")
else:
    print(f"Found {len(test_images)} test image(s) in {test_dir}.")
    
    # Run inference on test images
    for img_path in test_images:
        # Load and preprocess image identical to training
        raw_img = Image.open(img_path).convert('RGB')
        tensor_img = transform(raw_img).unsqueeze(0).to(device)  # Add batch dimension -> (1, 3, 224, 224)
        
        # Make prediction
        with torch.no_grad():
            prob = loaded_model(tensor_img).item()
            
        pred_class = "dog" if prob > 0.5 else "cat"
        confidence = prob if prob > 0.5 else (1.0 - prob)
        
        filename = os.path.basename(img_path)
        print(f"\nFile: {filename}")
        print(f"Prediction : {pred_class.upper()}")
        print(f"Raw Output : {prob:.4f}")
        print(f"Confidence : {confidence:.2%}")
        
        # Visualize the test image with prediction label
        plt.figure(figsize=(3.5, 3.5))
        plt.imshow(raw_img)
        plt.title(f"{filename}\nPred: {pred_class.upper()} ({confidence:.1%})")
        plt.axis('off')
        plt.show()


## 🌟 [Bonus Educational Section] CNN Architecture Upgrade

> **Why CNN over Flattened MLP?**
> In the baseline MLP above, flattening a $224 \times 224 \times 3$ image yields $150,528$ input features.
> Connecting this to a dense layer with $228$ neurons requires:
> $$\text{Parameters} = 150,528 \times 228 + 228 = 34,320,612 \text{ parameters}$$ 
> Over 34 million parameters on a small dataset can lead to heavy overfitting and memorization.
>
> A **Convolutional Neural Network (CNN)** uses spatial filters (`nn.Conv2d`) with weight sharing, reducing parameters down to under **50,000** while learning spatial edges, textures, and object shapes.

Run the cell below if you want to test and demonstrate this modern CNN alternative alongside your required MLP!

In [ ]:
# CELL 4: Optional Enhanced CNN Architecture
class CatDogCNN(nn.Module):
    def __init__(self):
        super(CatDogCNN, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 224 -> 112
            
            # Block 2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 112 -> 56
            
            # Block 3
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)   # 56 -> 28
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

cnn_model = CatDogCNN().to(device)
total_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f"CNN Model Initialized! Total Trainable Parameters: {total_params:,}")
print("(Notice: ~30,000 params vs. 34,320,000 in the flattened MLP!)")
